# Notebook 43 — Robust Engineering Preserves the Largest Connected Admissible Region

**Engineering statement**

> Robust engineering preserves the largest connected admissible region under perturbation, enabling fault-tolerant computation.

Notebook 37 established that computation requires a connected admissible region. This notebook takes the next step: it asks how that region survives drift, loss, detector noise, timing error, calibration changes, and architecture choices.

The goal is not to introduce new quantum formalism. The goal is to make the engineering logic explicit:

\[
\text{resources} \rightarrow \text{admissibility} \rightarrow \text{connected region} \rightarrow \text{robustness} \rightarrow \text{computation}.
\]

## Setup

This notebook creates reproducible figures and data files.

Outputs are written to:

```text
figures/
results/csv/
results/json/
results/43_outputs.zip
```

The final cell packages the figures and tables for download.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle, FancyArrowPatch
from matplotlib.collections import LineCollection
from collections import deque

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIG = ROOT / "figures"
RES = ROOT / "results"
CSV = RES / "csv"
JS = RES / "json"

for p in [FIG, RES, CSV, JS]:
    p.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

def savefig(fig, filename, dpi=220):
    path = FIG / filename
    fig.tight_layout()
    fig.savefig(path, dpi=dpi)
    print(f"saved: {path}")
    return path

def save_table(df, stem):
    csv_path = CSV / f"{stem}.csv"
    json_path = JS / f"{stem}.json"
    df.to_csv(csv_path, index=False)
    df.to_json(json_path, orient="records", indent=2)
    print(f"saved: {csv_path}")
    print(f"saved: {json_path}")

print("ROOT:", ROOT)
print("FIG:", FIG)
print("RES:", RES)

## 1. From operating point to robust region

A single operating point can be locally good and globally fragile.

A connected admissible region is stronger because small perturbations can move the system without immediately violating the active specifications.

Let

\[
\mathcal R
\]

denote the connected admissible region.

Let

\[
\delta(\mathcal R)
\]

denote the smallest perturbation that moves the system outside that region.

The engineering objective is therefore not simply

\[
\max \text{performance at one point},
\]

but rather

\[
\max \delta(\mathcal R)
\]

while preserving connectivity.

In [ ]:
def admissibility_landscape(n=320):
    x = np.linspace(0, 1, n)
    y = np.linspace(0, 1, n)
    X, Y = np.meshgrid(x, y)

    # A deliberately simple engineering landscape:
    # - not enough nonlinear drive on the left
    # - loss punishes the vertical direction
    # - overdrive/instability punishes the far right
    # - a broad connected region exists near moderate drive and low loss.
    squeezing_gate = 1 / (1 + np.exp(-15 * (X - 0.30)))
    loss_decay = np.exp(-2.65 * Y)
    overdrive = np.exp(-9.0 * np.maximum(X - 0.86, 0) ** 2)
    timing_penalty = np.exp(-4.3 * np.maximum(X + 0.70 * Y - 1.06, 0) ** 2)
    broad_region = np.exp(-((X - 0.62) ** 2 / 0.28 + (Y - 0.20) ** 2 / 0.20))

    Z = squeezing_gate * loss_decay * overdrive * timing_penalty * (0.52 + 0.48 * broad_region)
    Z = Z / Z.max()
    return x, y, X, Y, Z

x, y, X, Y, Z = admissibility_landscape()

target = (0.60, 0.22)

fig, ax = plt.subplots(figsize=(10, 6.3))
im = ax.imshow(Z, origin="lower", extent=[0, 1, 0, 1], aspect="auto", vmin=0, vmax=1)
cs = ax.contour(X, Y, Z, levels=[0.35, 0.55, 0.72], colors="black", linewidths=[1.5, 2.0, 2.8])
ax.clabel(cs, inline=True, fontsize=9)

ax.scatter([target[0]], [target[1]], s=240, zorder=5)
ax.annotate("design operating point", xy=target, xytext=(0.72, 0.34),
            arrowprops=dict(arrowstyle="->", linewidth=2.1), fontsize=11)

margin = Circle(target, 0.12, fill=False, linewidth=2.3, linestyle="--")
ax.add_patch(margin)

ax.text(0.50, 0.49, "largest connected\\nadmissible region", ha="center", fontsize=15)
ax.text(0.68, 0.18, "robust interior", fontsize=11)
ax.text(0.17, 0.82, "under-driven", ha="center", fontsize=11)
ax.text(0.84, 0.82, "unstable /\\nover-driven", ha="center", fontsize=11)
ax.text(0.87, 0.08, "loss-limited", ha="center", fontsize=11)

ax.set_title("Robustness Margin Around the Largest Connected Admissible Region", fontsize=17)
ax.set_xlabel("squeezing / nonlinear interaction strength")
ax.set_ylabel("optical loss")
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("admissibility score")

savefig(fig, "43_01_robustness_margin.png")
plt.show()

### Engineering observation

The blue point is not the whole design. It is only the selected operating point inside a larger region. The dashed circle is an illustrative robustness margin: small perturbations stay inside the admitted region, while larger perturbations cross a failure boundary.

This is why the notebook emphasizes **largest connected admissible region** rather than a single maximum.

## 2. Every perturbation has a failure boundary

Perturbations are directional.

A design may tolerate one kind of error while remaining fragile to another. The relevant robustness margin is the smallest distance to any active failure boundary.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 8.5))
ax.set_aspect("equal")
ax.axis("off")

center = np.array([0.52, 0.50])
safe_radius = 0.30
failure_radius = 0.45

ax.add_patch(Circle(center, failure_radius, fill=False, linewidth=2.2, alpha=0.35))
ax.add_patch(Circle(center, safe_radius, fill=True, alpha=0.10))
ax.add_patch(Circle(center, safe_radius, fill=False, linewidth=3.0))
ax.scatter([center[0]], [center[1]], s=280, zorder=5)

small_vectors = {
    "loss drift": np.array([-0.10, 0.15]),
    "phase drift": np.array([0.16, 0.10]),
    "timing error": np.array([-0.13, -0.13]),
    "detector noise": np.array([0.18, -0.10]),
}
large_vectors = {
    "calibration jump": np.array([0.36, 0.25]),
    "pump surge": np.array([0.41, -0.04]),
}

for label, v in small_vectors.items():
    ax.arrow(center[0], center[1], v[0], v[1],
             width=0.005, head_width=0.030, length_includes_head=True)
    ax.text(center[0] + 1.18*v[0], center[1] + 1.18*v[1], label, fontsize=10)

for label, v in large_vectors.items():
    ax.arrow(center[0], center[1], v[0], v[1],
             width=0.004, head_width=0.030, length_includes_head=True, alpha=0.60)
    ax.text(center[0] + 1.03*v[0], center[1] + 1.03*v[1], label, fontsize=10)

ax.text(center[0], center[1] - 0.06, "operating\npoint", ha="center", va="top", fontsize=10)
ax.text(center[0], center[1] + safe_radius + 0.04, "safe perturbation margin", ha="center", fontsize=11)
ax.text(center[0], center[1] - failure_radius - 0.08, "large perturbations cross a failure boundary", ha="center", fontsize=11)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Every Perturbation Has a Failure Boundary", fontsize=17)

savefig(fig, "43_02_perturbation_boundaries.png")
plt.show()

### Engineering observation

The shortest arrow to a failure boundary sets the practical margin. A design can have excellent squeezing and still fail because timing, detection, routing, or calibration has a smaller margin.

## 3. Region connectivity under perturbation

Robustness is not merely area. Connectivity matters.

A region can retain significant area while becoming disconnected. A disconnected admitted set is harder to use because computation requires paths, routing, measurement order, and stable feed-forward.

In [ ]:
def gaussian_blob(X, Y, cx, cy, sx, sy, amp=1.0):
    return amp * np.exp(-((X-cx)**2/(2*sx**2) + (Y-cy)**2/(2*sy**2)))

def clean_region(kind="baseline", n=220):
    x = np.linspace(0, 1, n)
    y = np.linspace(0, 1, n)
    X, Y = np.meshgrid(x, y)

    if kind == "baseline":
        Z = (
            gaussian_blob(X, Y, 0.45, 0.32, 0.18, 0.18, 1.00)
            + gaussian_blob(X, Y, 0.62, 0.34, 0.18, 0.18, 0.95)
            + gaussian_blob(X, Y, 0.52, 0.18, 0.25, 0.12, 0.80)
        )
    elif kind == "shrunk":
        Z = (
            gaussian_blob(X, Y, 0.48, 0.30, 0.14, 0.14, 1.00)
            + gaussian_blob(X, Y, 0.64, 0.29, 0.13, 0.13, 0.85)
        ) * np.exp(-0.90 * Y)
    elif kind == "disconnected":
        Z = (
            gaussian_blob(X, Y, 0.28, 0.28, 0.10, 0.11, 1.00)
            + gaussian_blob(X, Y, 0.52, 0.36, 0.11, 0.11, 1.00)
            + gaussian_blob(X, Y, 0.80, 0.24, 0.10, 0.10, 1.00)
        )
        # carve gaps between islands
        Z -= 0.48 * gaussian_blob(X, Y, 0.41, 0.32, 0.06, 0.13, 1.0)
        Z -= 0.46 * gaussian_blob(X, Y, 0.66, 0.30, 0.07, 0.13, 1.0)
        Z = np.clip(Z, 0, None)
    else:
        raise ValueError(kind)

    Z = Z / Z.max()
    return x, y, X, Y, Z

def connected_components(mask):
    visited = np.zeros_like(mask, dtype=bool)
    h, w = mask.shape
    sizes = []
    for i in range(h):
        for j in range(w):
            if mask[i, j] and not visited[i, j]:
                q = deque([(i, j)])
                visited[i, j] = True
                size = 0
                while q:
                    a, b = q.popleft()
                    size += 1
                    for da, db in [(1,0),(-1,0),(0,1),(0,-1)]:
                        na, nb = a + da, b + db
                        if 0 <= na < h and 0 <= nb < w and mask[na, nb] and not visited[na, nb]:
                            visited[na, nb] = True
                            q.append((na, nb))
                sizes.append(size)
    sizes.sort(reverse=True)
    return sizes

cases = [("baseline", "Baseline"), ("shrunk", "Shrunk"), ("disconnected", "Disconnected")]
metrics = []

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharex=True, sharey=True)

threshold = 0.50
for ax, (kind, title) in zip(axes, cases):
    x, y, X, Y, Zr = clean_region(kind)
    mask = Zr >= threshold
    comps = connected_components(mask)
    area = mask.mean()
    largest = comps[0] / mask.size if comps else 0.0
    n_components = len(comps)

    metrics.append({
        "case": kind,
        "admitted_area_fraction": area,
        "largest_component_fraction": largest,
        "component_count": n_components,
    })

    ax.imshow(Zr, origin="lower", extent=[0, 1, 0, 1], aspect="auto", vmin=0, vmax=1)
    ax.contour(X, Y, Zr, levels=[threshold], colors="black", linewidths=2.4)
    ax.set_title(title)
    ax.set_xlabel("drive")
    if ax is axes[0]:
        ax.set_ylabel("loss")
    ax.text(0.05, 0.90, f"components: {n_components}", transform=ax.transAxes, fontsize=10)
    ax.text(0.05, 0.82, f"largest: {largest:.2f}", transform=ax.transAxes, fontsize=10)

fig.suptitle("Region Connectivity Under Perturbation", fontsize=17, y=1.04)
savefig(fig, "43_03_region_connectivity.png")
plt.show()

region_metrics = pd.DataFrame(metrics)
save_table(region_metrics, "43_region_connectivity")
region_metrics

### Engineering observation

The disconnected case may still contain admitted resources. The problem is that those resources no longer form a useful connected operating region. Notebook 43 therefore treats **largest connected component** as more meaningful than raw admitted area.

## 4. Architecture comparison scorecard

Different architectures preserve different combinations of:

- region size
- operating margin
- connectivity
- calibration stability
- timing support

A useful architecture comparison should not collapse all of these into one opaque score too early.

In [ ]:
scorecard = pd.DataFrame({
    "architecture": ["single resonator", "coupled resonators", "programmable mesh", "hybrid chip"],
    "region_size": [2, 3, 4, 5],
    "margin": [1, 2, 4, 5],
    "connectivity": [2, 3, 5, 5],
    "calibration": [4, 3, 3, 4],
    "timing": [1, 2, 3, 5],
})
scorecard["mean_score"] = scorecard[["region_size","margin","connectivity","calibration","timing"]].mean(axis=1)

fig, ax = plt.subplots(figsize=(11, 5.7))
ax.axis("off")

columns = ["Architecture", "Region", "Margin", "Connectivity", "Calibration", "Timing"]
rows = []
for _, row in scorecard.iterrows():
    rows.append([
        row["architecture"],
        "●" * int(row["region_size"]),
        "●" * int(row["margin"]),
        "●" * int(row["connectivity"]),
        "●" * int(row["calibration"]),
        "●" * int(row["timing"]),
    ])

table = ax.table(cellText=rows, colLabels=columns, cellLoc="center", loc="center")
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.05, 1.8)

for (r, c), cell in table.get_celld().items():
    cell.set_linewidth(1.2)
    if r == 0:
        cell.set_text_props(weight="bold")
    if c == 0 and r > 0:
        cell.set_text_props(ha="left")

ax.set_title("Architecture Comparison: Region, Margin, and Connectivity", fontsize=17, pad=18)
ax.text(0.5, -0.06, "More dots indicate stronger support for preserving the largest connected admissible region.",
        ha="center", transform=ax.transAxes, fontsize=11)

save_table(scorecard, "43_architecture_scorecard")
savefig(fig, "43_04_architecture_scorecard.png")
plt.show()
scorecard

### Engineering observation

The hybrid chip scores well because it combines multiple supports: squeezing, routing, detection, timing, and stabilization. The single resonator may preserve stability but has limited margin and feed-forward capability.

## 5. Design landscape: prefer plateaus over ridges

A narrow optimum can look excellent in a static benchmark and still fail under perturbation.

A broad plateau is often more valuable because it preserves performance over a region.

In [ ]:
x = np.linspace(-1.2, 1.2, 320)
y = np.linspace(-1.2, 1.2, 320)
X, Y = np.meshgrid(x, y)

# Smooth plateau: high over a broad region.
plateau_x = np.exp(-((X + 0.20) ** 8) / 0.18)
plateau_y = np.exp(-((Y + 0.05) ** 8) / 0.08)
plateau = 0.72 * plateau_x * plateau_y

# Narrow ridge: taller but fragile.
ridge = 1.10 * np.exp(-((X - 0.53) ** 2 / 0.018 + (Y + 0.02) ** 2 / 0.34))

# Instability cliff at high y.
cliff = 1 / (1 + np.exp(9 * (Y - 0.70)))

landscape = (plateau + ridge) * cliff
landscape = landscape / landscape.max()

fig, ax = plt.subplots(figsize=(9.5, 6.2))
im = ax.imshow(landscape, origin="lower", extent=[x.min(), x.max(), y.min(), y.max()], aspect="auto", vmin=0, vmax=1)
cs = ax.contour(X, Y, landscape, levels=[0.35, 0.55, 0.75], colors="black", linewidths=[1.5, 2.0, 2.6])
ax.clabel(cs, inline=True, fontsize=9)

ax.text(-0.55, 0.05, "safe plateau", fontsize=14)
ax.text(0.54, -0.16, "narrow ridge", fontsize=14)
ax.text(0.32, 0.84, "unstable cliff", fontsize=13)
ax.annotate("good engineering\nwidens this plateau", xy=(-0.36, 0.17), xytext=(-1.05, 0.70),
            arrowprops=dict(arrowstyle="->", linewidth=2.2), fontsize=12)

ax.set_title("Design Landscape: Prefer a Plateau Over a Ridge", fontsize=17)
ax.set_xlabel("control parameter 1")
ax.set_ylabel("control parameter 2")
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("relative computation support")

savefig(fig, "43_05_design_landscape.png")
plt.show()

### Engineering observation

This figure is intentionally generic. It applies to photonic cluster-state design, control systems, optimization, and hardware calibration. A narrow ridge may maximize a benchmark; a plateau preserves operation.

## 6. Failure tree

Logical failure is usually downstream.

The earlier engineering failure is that perturbations disconnect or shrink the admissible region until no fault-tolerant path remains.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.axis("off")

leaves = [
    ("loss", (0.10, 0.82)),
    ("phase drift", (0.10, 0.64)),
    ("detector noise", (0.10, 0.46)),
    ("timing jitter", (0.10, 0.28)),
    ("feed-forward delay", (0.10, 0.10)),
]

mid_label = "connected admissible\nregion disconnects"
mid_pos = (0.55, 0.46)
out_label = "logical failure"
out_pos = (0.88, 0.46)

for label, pos in leaves:
    ax.add_patch(Rectangle((pos[0]-0.085, pos[1]-0.045), 0.17, 0.09, fill=False, linewidth=1.8))
    ax.text(*pos, label, ha="center", va="center", fontsize=10)
    ax.annotate("", xy=(mid_pos[0]-0.16, mid_pos[1]), xytext=(pos[0]+0.09, pos[1]),
                arrowprops=dict(arrowstyle="->", linewidth=1.4, alpha=0.75))

ax.add_patch(Rectangle((mid_pos[0]-0.16, mid_pos[1]-0.075), 0.32, 0.15, fill=False, linewidth=2.4))
ax.text(*mid_pos, mid_label, ha="center", va="center", fontsize=11)

ax.add_patch(Rectangle((out_pos[0]-0.09, out_pos[1]-0.045), 0.18, 0.09, fill=False, linewidth=1.9))
ax.text(*out_pos, out_label, ha="center", va="center", fontsize=11)
ax.annotate("", xy=(out_pos[0]-0.10, out_pos[1]), xytext=(mid_pos[0]+0.17, mid_pos[1]),
            arrowprops=dict(arrowstyle="->", linewidth=2.2))

ax.set_title("Failure Tree: Logical Failure Follows Disconnection of the Region", fontsize=17)
ax.text(0.5, -0.03, "robustness protects the intermediate engineering object before computation fails",
        ha="center", transform=ax.transAxes, fontsize=11)

savefig(fig, "43_06_failure_tree.png")
plt.show()

## 7. Engineering priority

A pie chart implies measured percentages. For this notebook, the better representation is a priority ranking.

The ranking says where effort should go if the objective is to preserve the largest connected admissible region.

In [ ]:
priority = pd.DataFrame({
    "priority": [
        "Preserve admissibility",
        "Increase operating margin",
        "Improve detection",
        "Stabilize calibration",
        "Reduce timing error",
        "Local optimization",
    ],
    "relative_priority": [10.0, 8.5, 7.2, 6.1, 5.6, 3.8],
})

fig, ax = plt.subplots(figsize=(9, 5.5))
ypos = np.arange(len(priority))[::-1]
ax.barh(ypos, priority["relative_priority"])
ax.set_yticks(ypos)
ax.set_yticklabels(priority["priority"])
ax.set_xlim(0, 10.8)
ax.set_xlabel("relative engineering priority")
ax.set_title("Engineering Priority: Effort Follows Robustness Margin", fontsize=17)
ax.grid(axis="x", alpha=0.25)

for y, value in zip(ypos, priority["relative_priority"]):
    ax.text(value + 0.15, y, f"{value:g}", va="center", fontsize=10)

save_table(priority, "43_engineering_priority")
savefig(fig, "43_07_engineering_priority.png")
plt.show()
priority

### Engineering observation

Local optimization is deliberately last. Once the largest connected admissible region is preserved, optimization can refine performance. But optimizing before admissibility is preserved can make the design brittle.

## 8. Robustness pipeline

Notebook 37 identified the connected admissible region.

Notebook 43 adds the preservation step:

\[
	ext{connected admissible region} ightarrow 	ext{largest connected admissible region under perturbation}.
\]

In [ ]:
fig, ax = plt.subplots(figsize=(13.5, 3.8))
ax.axis("off")

boxes = [
    "Physics",
    "Generated\nresources",
    "Admissibility",
    "Connected\nadmissible region",
    "Robustness",
    "Largest connected\nadmissible region",
    "Fault-tolerant\ncomputation",
]
xs = np.linspace(0.06, 0.94, len(boxes))
y = 0.52
w = 0.115
h = 0.32

for i, (x0, label) in enumerate(zip(xs, boxes)):
    ax.add_patch(Rectangle((x0 - w/2, y - h/2), w, h, fill=False, linewidth=2.0))
    ax.text(x0, y, label, ha="center", va="center", fontsize=10)
    if i < len(boxes) - 1:
        ax.annotate("", xy=(xs[i+1] - w/2 - 0.01, y), xytext=(x0 + w/2 + 0.01, y),
                    arrowprops=dict(arrowstyle="->", linewidth=2.0))

ax.text(0.5, 0.08, "Each stage specifies what must be preserved by the next.", ha="center", fontsize=12)
ax.set_title("Robustness Pipeline: Preserve the Region, Not Just the Point", fontsize=17)

savefig(fig, "43_08_robustness_pipeline.png")
plt.show()

## 9. Engineering summary

Robustness turns admissibility into an engineering method.

The central object is not a single parameter and not a single metric. The object is the connected region that remains useful under perturbation.

In [ ]:
summary = pd.DataFrame(
    [
        {
            "Stage": "Physics",
            "Question": "What can exist?",
            "Maintained quantity": "Optical resources",
            "Engineering role": "Generate candidate resources",
        },
        {
            "Stage": "Admissibility",
            "Question": "What survives?",
            "Maintained quantity": "Admitted resources",
            "Engineering role": "Filter failed resources",
        },
        {
            "Stage": "Region",
            "Question": "Where can computation occur?",
            "Maintained quantity": "Connected admissible region",
            "Engineering role": "Preserve connectivity",
        },
        {
            "Stage": "Robustness",
            "Question": "How much perturbation is tolerated?",
            "Maintained quantity": "Largest connected admissible region",
            "Engineering role": "Maintain stability margin",
        },
        {
            "Stage": "Optimization",
            "Question": "Which design widens the region?",
            "Maintained quantity": "Operating margin",
            "Engineering role": "Improve after preserving admissibility",
        },
        {
            "Stage": "Computation",
            "Question": "Which logical operations remain?",
            "Maintained quantity": "Fault-tolerant execution",
            "Engineering role": "Support logical computation",
        },
    ]
)

fig, ax = plt.subplots(figsize=(15.5, 4.8))
ax.axis("off")
table = ax.table(
    cellText=summary.values,
    colLabels=summary.columns,
    loc="center",
    cellLoc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(8.5)
table.scale(1.08, 1.72)
for (r, c), cell in table.get_celld().items():
    cell.set_linewidth(1.15)
    if r == 0:
        cell.set_text_props(weight="bold")

ax.set_title("Engineering Summary: Robustness Preserves Admissible Computation", fontsize=17, pad=22)
ax.text(0.5, -0.10, "Robust engineering preserves the largest connected admissible region, not merely the operating point.",
        ha="center", transform=ax.transAxes, fontsize=11)

save_table(summary, "43_summary")
savefig(fig, "43_09_summary_table.png")
plt.show()
summary

## 10. Quantitative robustness metrics

Notebook 47 can extend this notebook by making the metrics more explicit.

Useful quantities include:

- admitted area
- largest connected component size
- component count
- distance to the nearest failure boundary
- perturbation survival probability
- architecture robustness score

This notebook already computes admitted area and largest connected component size for the region-survival example.

In [ ]:
# Combine exported metric tables into one compact review table.
review = {
    "region_connectivity": region_metrics.to_dict(orient="records"),
    "architecture_scorecard": scorecard.to_dict(orient="records"),
    "engineering_priority": priority.to_dict(orient="records"),
}

review_path = JS / "43_review_bundle.json"
with open(review_path, "w", encoding="utf-8") as f:
    json.dump(review, f, indent=2)

print(f"saved: {review_path}")

## Takeaways

1. Robustness is preservation under perturbation.
2. The central engineering object is the **largest connected admissible region**.
3. Connectivity is more important than raw area.
4. Broad plateaus beat narrow peaks when the design must survive drift.
5. Architecture comparison should evaluate region size, margin, and connectivity together.
6. Logical failure usually follows an earlier engineering failure: disconnection of the admitted region.

## Next notebook

A natural next notebook is:

```text
47 — Quantifying Robustness Margins
```

It can turn the qualitative diagrams here into explicit metrics and architecture comparisons.

## Export and download bundle

In [ ]:
zip_path = RES / "43_outputs.zip"

files_to_zip = [
    FIG / "43_01_robustness_margin.png",
    FIG / "43_02_perturbation_boundaries.png",
    FIG / "43_03_region_connectivity.png",
    FIG / "43_04_architecture_scorecard.png",
    FIG / "43_05_design_landscape.png",
    FIG / "43_06_failure_tree.png",
    FIG / "43_07_engineering_priority.png",
    FIG / "43_08_robustness_pipeline.png",
    FIG / "43_09_summary_table.png",
    CSV / "43_region_connectivity.csv",
    CSV / "43_architecture_scorecard.csv",
    CSV / "43_engineering_priority.csv",
    CSV / "43_summary.csv",
    JS / "43_region_connectivity.json",
    JS / "43_architecture_scorecard.json",
    JS / "43_engineering_priority.json",
    JS / "43_summary.json",
    JS / "43_review_bundle.json",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        if file.exists():
            z.write(file, file.relative_to(ROOT))

print(f"Created: {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))